# Evaluate the balanced activation pipeline

This notebook does not train models or change checkpoints and feature files.
Configure the result path in the next cell before evaluation.

In [1]:
from pathlib import Path
from functools import lru_cache

from IPython.display import display

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Correct this name to balanced_activation_pipeline if that is your folder name.
RESULTS_ROOT = Path(
    "/mnt/c/Users/anton/Desktop/balanced_activation_pipeline"
)
EVALUATION_ROOT = RESULTS_ROOT / "evaluation"
TASKS = [f"{element}_FEFF" for element in "Ti V Cr Mn Fe Co Ni Cu".split()]
ELEMENTS = [task.removesuffix("_FEFF") for task in TASKS]
SPLITS = ("train", "val", "test")
VARIANTS = ("silu", "gelu", "selu")
INPUT_DIM, OUTPUT_DIM = 64, 141
PREDICTION_BATCH_SIZE = 4096
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
display(pd.DataFrame({"device": [str(DEVICE)]}))

,device
0,cpu


In [2]:
required_feature_files = [
    f"{task}_{split}_{suffix}.txt"
    for task in TASKS
    for split in SPLITS
    for suffix in ("X", "y")
]

def has_complete_features(variant):
    directory = RESULTS_ROOT / variant / "features"
    return directory.is_dir() and all((directory / filename).is_file() for filename in required_feature_files)

found_variants = [variant for variant in VARIANTS if has_complete_features(variant)]
skipped_variants = [variant for variant in VARIANTS if (RESULTS_ROOT / variant).is_dir() and variant not in found_variants]
display(pd.DataFrame({"found_complete_variants": [", ".join(found_variants)], "skipped_incomplete_variants": [", ".join(skipped_variants)]}))
if not found_variants:
    raise FileNotFoundError(f"No complete variants found under {RESULTS_ROOT}")

@lru_cache(maxsize=None)
def load_features(variant, task, split):
    directory = RESULTS_ROOT / variant / "features"
    x_path = directory / f"{task}_{split}_X.txt"
    y_path = directory / f"{task}_{split}_y.txt"
    if not x_path.is_file() or not y_path.is_file():
        raise FileNotFoundError(f"Missing exported features for {variant}, {task}, {split}")
    x = np.atleast_2d(np.loadtxt(x_path, dtype=np.float32))
    y = np.atleast_2d(np.loadtxt(y_path, dtype=np.float32))
    if x.shape != (len(x), INPUT_DIM):
        raise ValueError(f"Invalid X shape in {x_path}: {x.shape}")
    if y.shape != (len(y), OUTPUT_DIM):
        raise ValueError(f"Invalid y shape in {y_path}: {y.shape}")
    if x.shape[0] != y.shape[0]:
        raise ValueError(f"Row count mismatch for {task} {split}: {x.shape[0]} versus {y.shape[0]}")
    if not np.isfinite(x).all() or not np.isfinite(y).all():
        raise ValueError(f"Non-finite feature or target value for {task} {split}")
    return x, y

def checkpoint_state(path, prefix):
    checkpoint = torch.load(path, map_location="cpu")
    state = checkpoint.get("state_dict", checkpoint)
    selected = {key.removeprefix(prefix): value for key, value in state.items() if key.startswith(prefix)}
    if not selected:
        raise ValueError(f"No weights with prefix {prefix!r} in {path}")
    return selected

class Head(nn.Sequential):
    def __init__(self, variant, dropout=0.10):
        layers = []
        dimensions = (INPUT_DIM, 500, 500, 550, OUTPUT_DIM)
        for index, (left, right) in enumerate(zip(dimensions, dimensions[1:])):
            layers.append(nn.Linear(left, right))
            if index < 3:
                if variant == "selu":
                    layers.extend((nn.SELU(), nn.Dropout(dropout)))
                elif variant == "silu":
                    layers.extend((nn.BatchNorm1d(right), nn.SiLU(), nn.Dropout(dropout)))
                elif variant == "gelu":
                    layers.extend((nn.BatchNorm1d(right), nn.GELU(), nn.Dropout(dropout)))
                else:
                    raise ValueError(f"Unsupported variant: {variant}")
            else:
                layers.append(nn.Softplus())
        super().__init__(*layers)

def load_head(path, variant, prefix):
    head = Head(variant).to(DEVICE)
    head.load_state_dict(checkpoint_state(path, prefix), strict=True)
    return head.eval()


,found_complete_variants,skipped_incomplete_variants
0,"silu, gelu, selu",


In [3]:
def predict(head, x):
    loader = DataLoader(TensorDataset(torch.from_numpy(x)), batch_size=PREDICTION_BATCH_SIZE)
    predictions = []
    with torch.inference_mode():
        for (batch,) in loader:
            predictions.append(head(batch.to(DEVICE)).cpu().numpy())
    return np.concatenate(predictions)

def metrics(variant, task, split, head, family, checkpoint, **settings):
    x, y = load_features(variant, task, split)
    _, train_y = load_features(variant, task, "train")
    prediction = predict(head, x)
    per_spectrum = np.mean((prediction - y) ** 2, axis=1)
    baseline_prediction = train_y.mean(axis=0)
    baseline_per_spectrum = np.mean((baseline_prediction - y) ** 2, axis=1)
    median_mse = float(np.median(per_spectrum))
    row = {
        "variant": variant, "family": family, "task": task, "element": task.removesuffix("_FEFF"),
        "split": split, "checkpoint": str(checkpoint), "aggregate_mse": float(per_spectrum.mean()),
        "median_per_spectrum_mse": median_mse,
        "baseline_median_mse": float(np.median(baseline_per_spectrum)),
        "eta": float(np.median(baseline_per_spectrum) / median_mse),
    }
    row.update(settings)
    return row

def complete_candidates(variant):
    root = RESULTS_ROOT / variant
    candidates = []
    e2e = root / "e2e" / "best.ckpt"
    if e2e.is_file():
        candidates.append(("e2e", e2e, {}))
    for path in sorted((root / "universal").glob("batch_*/seed_*/best.ckpt")):
        candidates.append(("universal", path, {"batch_size": path.parents[1].name.removeprefix("batch_"), "source_seed": path.parent.name.removeprefix("seed_")}))
    for path in sorted((root / "tuned").glob("source_batch_*/source_seed_*/**/seed_*/best.ckpt")):
        element = path.parents[1].name
        candidates.append(("tuned", path, {"batch_size": path.parents[3].name.removeprefix("source_batch_"), "source_seed": path.parents[2].name.removeprefix("source_seed_"), "tuned_seed": path.parent.name.removeprefix("seed_"), "tuned_element": element}))
    return candidates

candidates = {variant: complete_candidates(variant) for variant in found_variants}
checkpoint_counts = (
    pd.DataFrame(
        [
            {"variant": variant, **pd.Series([family for family, _, _ in rows]).value_counts().to_dict()}
            for variant, rows in candidates.items()
        ]
    )
    .reindex(columns=["variant", "e2e", "universal", "tuned"], fill_value=0)
    .fillna(0)
    .astype({"e2e": int, "universal": int, "tuned": int})
)
display(checkpoint_counts)
if not any(candidates.values()):
    raise FileNotFoundError(f"No complete best.ckpt files found under {RESULTS_ROOT}")


,variant,e2e,universal,tuned
0,silu,1,9,216
1,gelu,1,9,216
2,selu,1,9,216


In [4]:
validation_rows = []
for variant in found_variants:
    for family, checkpoint, settings in candidates[variant]:
        if family == "tuned":
            tasks = [f"{settings['tuned_element']}_FEFF"]
        else:
            tasks = TASKS
        prefix = "model.head." if family == "e2e" else "head."
        head = load_head(checkpoint, variant, prefix)
        for task in tasks:
            validation_rows.append(metrics(variant, task, "val", head, family, checkpoint, **settings))
validation_metrics = pd.DataFrame(validation_rows)
if validation_metrics.empty:
    raise RuntimeError("No validation metrics were calculated")
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
validation_metrics.to_csv(EVALUATION_ROOT / "validation_metrics.csv", index=False)
validation_counts = validation_metrics.groupby("family").size().rename("rows").reset_index()
display(validation_counts)

,family,rows
0,e2e,24
1,tuned,648
2,universal,216


In [5]:
# Select checkpoints from validation eta only.
def select_group(frame, columns):
    identity = ["family", *columns, "checkpoint"]
    grouped = frame.groupby(identity, dropna=False, as_index=False)["eta"].mean()
    extra = [name for name in ("tuned_seed",) if name in frame and name not in grouped]
    if extra:
        metadata = frame.drop_duplicates(identity)[identity + extra]
        grouped = grouped.merge(metadata, on=identity, how="left")
    return grouped.sort_values("eta", ascending=False).drop_duplicates(["family", *columns], keep="first")

e2e_val = validation_metrics[validation_metrics.family == "e2e"].copy()
e2e_selection = select_group(e2e_val, ["variant"])
universal_val = validation_metrics[validation_metrics.family == "universal"].copy()
universal_selection = select_group(universal_val, ["variant", "batch_size"])
tuned_val = validation_metrics[validation_metrics.family == "tuned"].copy()
tuned_selection = select_group(tuned_val, ["variant", "batch_size", "source_seed", "tuned_element"])
selection = pd.concat([e2e_selection, universal_selection, tuned_selection], ignore_index=True)
selection.to_csv(EVALUATION_ROOT / "validation_selection.csv", index=False)
display(selection.sort_values(["family", "variant"]).reset_index(drop=True))

,family,variant,checkpoint,eta,tuned_seed,batch_size,source_seed,tuned_element
0,e2e,gelu,/mnt/c/Users/anton/Desktop/balanced_activation...,17.683350,NaN,NaN,NaN,NaN
1,e2e,selu,/mnt/c/Users/anton/Desktop/balanced_activation...,2.306882,NaN,NaN,NaN,NaN
2,e2e,silu,/mnt/c/Users/anton/Desktop/balanced_activation...,17.293113,NaN,NaN,NaN,NaN
3,tuned,gelu,/mnt/c/Users/anton/Desktop/balanced_activation...,42.919598,145,128,45,Mn
4,tuned,gelu,/mnt/c/Users/anton/Desktop/balanced_activation...,42.364636,146,128,44,Mn
...,...,...,...,...,...,...,...,...
223,universal,selu,/mnt/c/Users/anton/Desktop/balanced_activation...,1.683560,NaN,64,NaN,NaN
224,universal,selu,/mnt/c/Users/anton/Desktop/balanced_activation...,1.638996,NaN,192,NaN,NaN
225,universal,silu,/mnt/c/Users/anton/Desktop/balanced_activation...,19.632554,NaN,192,NaN,NaN
226,universal,silu,/mnt/c/Users/anton/Desktop/balanced_activation...,19.574083,NaN,128,NaN,NaN


In [6]:
test_rows = []
for row in selection.to_dict(orient="records"):
    checkpoint = Path(row["checkpoint"])
    variant, family = row["variant"], row["family"]
    prefix = "model.head." if family == "e2e" else "head."
    head = load_head(checkpoint, variant, prefix)
    tasks = [f"{row['tuned_element']}_FEFF"] if family == "tuned" else TASKS
    settings = {key: row[key] for key in ("batch_size", "source_seed", "tuned_seed", "tuned_element") if key in row and pd.notna(row[key])}
    for task in tasks:
        test_rows.append(metrics(variant, task, "test", head, family, checkpoint, **settings))
test_metrics = pd.DataFrame(test_rows)
test_metrics.to_csv(EVALUATION_ROOT / "test_metrics.csv", index=False)
summary_columns = ["variant", "family"]
test_summary = test_metrics.groupby(summary_columns, as_index=False)[["aggregate_mse", "median_per_spectrum_mse", "baseline_median_mse", "eta"]].mean()
test_summary.to_csv(EVALUATION_ROOT / "test_summary.csv", index=False)
display(test_metrics)
display(test_summary)

per_element_eta = (
    test_metrics.groupby(["family", "element", "variant"], as_index=False)["eta"]
    .mean()
    .pivot(index=["family", "element"], columns="variant", values="eta")
    .reindex(columns=found_variants)
    .reset_index()
)
per_element_eta.to_csv(EVALUATION_ROOT / "test_eta_by_element.csv", index=False)
display(per_element_eta)

activation_average_eta = (
    per_element_eta.melt(
        id_vars=["family", "element"],
        var_name="variant",
        value_name="eta",
    )
    .dropna(subset=["eta"])
    .groupby(["family", "variant"], as_index=False)["eta"]
    .mean()
)
activation_average_eta.to_csv(EVALUATION_ROOT / "test_eta_activation_average.csv", index=False)
display(activation_average_eta)

,variant,family,task,element,split,checkpoint,aggregate_mse,median_per_spectrum_mse,baseline_median_mse,eta,batch_size,source_seed,tuned_seed,tuned_element
0,gelu,e2e,Ti_FEFF,Ti,test,/mnt/c/Users/anton/Desktop/balanced_activation...,0.008759,0.004594,0.048614,10.582673,NaN,NaN,NaN,NaN
1,gelu,e2e,V_FEFF,V,test,/mnt/c/Users/anton/Desktop/balanced_activation...,0.007118,0.004590,0.050749,11.056489,NaN,NaN,NaN,NaN
2,gelu,e2e,Cr_FEFF,Cr,test,/mnt/c/Users/anton/Desktop/balanced_activation...,0.005745,0.003245,0.056319,17.354889,NaN,NaN,NaN,NaN
3,gelu,e2e,Mn_FEFF,Mn,test,/mnt/c/Users/anton/Desktop/balanced_activation...,0.003339,0.001724,0.045599,26.446867,NaN,NaN,NaN,NaN
4,gelu,e2e,Fe_FEFF,Fe,test,/mnt/c/Users/anton/Desktop/balanced_activation...,0.006515,0.001815,0.028778,15.855861,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,selu,tuned,Ti_FEFF,Ti,test,/mnt/c/Users/anton/Desktop/balanced_activation...,0.048477,0.045485,0.048614,1.068790,64,44,146,Ti
308,selu,tuned,Ti_FEFF,Ti,test,/mnt/c/Users/anton/Desktop/balanced_activation...,0.051083,0.046969,0.048614,1.035010,128,44,147,Ti
309,selu,tuned,Ti_FEFF,Ti,test,/mnt/c/Users/anton/Desktop/balanced_activation...,0.051135,0.047448,0.048614,1.024565,64,45,145,Ti
310,selu,tuned,Ti_FEFF,Ti,test,/mnt/c/Users/anton/Desktop/balanced_activation...,0.050644,0.047587,0.048614,1.021582,192,44,146,Ti


,variant,family,aggregate_mse,median_per_spectrum_mse,baseline_median_mse,eta
0,gelu,e2e,0.005033,0.002517,0.037263,17.037855
1,gelu,tuned,0.004887,0.002268,0.037263,19.365307
2,gelu,universal,0.004824,0.002261,0.037263,19.007091
3,selu,e2e,0.020859,0.018609,0.037263,2.251751
4,selu,tuned,0.023230,0.020570,0.037263,2.137915
5,selu,universal,0.026201,0.023731,0.037263,1.628384
6,silu,e2e,0.005049,0.002503,0.037263,16.961662
7,silu,tuned,0.004868,0.002293,0.037263,19.360194
8,silu,universal,0.004864,0.002287,0.037263,18.815739


variant,family,element,silu,gelu,selu
0,e2e,Co,26.665563,28.884432,2.409530
1,e2e,Cr,18.363242,17.354889,4.035014
2,e2e,Cu,7.430294,7.106139,0.863408
3,e2e,Fe,14.636032,15.855861,2.341832
4,e2e,Mn,29.522516,26.446867,3.393043
5,e2e,Ni,17.305214,19.015488,2.057235
6,e2e,Ti,10.419251,10.582673,1.230740
7,e2e,V,11.351183,11.056489,1.683202
8,tuned,Co,32.515886,33.415887,2.306334
9,tuned,Cr,20.404892,19.157676,2.422071


,family,variant,eta
0,e2e,gelu,17.037855
1,e2e,selu,2.251751
2,e2e,silu,16.961662
3,tuned,gelu,19.365307
4,tuned,selu,2.137915
5,tuned,silu,19.360194
6,universal,gelu,19.007091
7,universal,selu,1.628384
8,universal,silu,18.815739
